<a href="https://colab.research.google.com/github/shirin6767saleh/code-/blob/Fnew/checktable.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#$\theta=0$

In [21]:
import numpy as np

# ----------------------------
# Parameters
# ----------------------------
m = 3
N = 4 * m + 2
w = np.exp(-2j * np.pi / N)

# ----------------------------
# Define vector d
# ----------------------------
d = np.zeros(2 * m + 2, dtype=complex)
d[0] = 1 / np.sqrt(2)
d[2 * m + 1] = 1 / np.sqrt(2)
d[1:2 * m + 1] = 1

# ----------------------------
# Compute C and S matrices
# ----------------------------
C = np.zeros((2 * m + 2, 2 * m + 2), dtype=complex)
for k in range(1, 2 * m + 3):
    for l in range(1, 2 * m + 3):
        C[k - 1, l - 1] = (1 / np.sqrt(N)) * d[k - 1] * d[l - 1] * np.cos((k - 1) * (l - 1) * 2 * np.pi / N)

S = np.zeros((2 * m, 2 * m), dtype=complex)
for k in range(1, 2 * m + 1):
    for l in range(1, 2 * m + 1):
        S[k - 1, l - 1] = (1 / np.sqrt(N)) * np.sin(k * l * 2 * np.pi / N)

# ----------------------------
# Define X, Y, Z, T
# ----------------------------
X = C + 0.5 * np.eye(2 * m + 2)
Y = C - 0.5 * np.eye(2 * m + 2)
Z = S + 0.5 * np.eye(2 * m)
T = S - 0.5 * np.eye(2 * m)

# ----------------------------
# Construct DFT matrix F
# ----------------------------
F = np.zeros((N, N), dtype=complex)
for k in range(1, N + 1):
    for l in range(1, N + 1):
        F[k - 1, l - 1] = (1 / np.sqrt(N)) * w ** ((k - 1) * (l - 1))

# ----------------------------
# Helper matrices
# ----------------------------
J = np.flip(np.eye(2 * m), axis=0)  # flip rows (like MATLAB flipud)

X1 = X[:, :m + 1]
Y1 = Y[:, :m + 1]
Z1 = Z[:, :m]
T1 = T[:, :m]

X0 = X1[1:2 * m + 1, :]
Y0 = Y1[1:2 * m + 1, :]

# ----------------------------
# Build extended matrices (MATLAB equivalent)
# ----------------------------
X2 = np.vstack([
    np.sqrt(2) * X1[0:1, :],
    X0,
    np.sqrt(2) * X1[2*m+1:2*m+2, :],
    J @ X0
])

Y2 = np.vstack([
    np.sqrt(2) * Y1[0:1, :],
    Y0,
    np.sqrt(2) * Y1[2*m+1:2*m+2, :],
    J @ Y0
])

Z2 = np.vstack([
    np.zeros((1, m)),
    Z1,
    np.zeros((1, m)),
    -J @ Z1
])

T2 = np.vstack([
    np.zeros((1, m)),
    T1,
    np.zeros((1, m)),
    -J @ T1
])

# ----------------------------
# Combine all into V
# ----------------------------
V = np.hstack([X2, Y2, T2, Z2])

# ----------------------------
# Define diagonal matrix D
# ----------------------------
D = np.zeros((N, N), dtype=complex)

# first two blocks
for k in range(m + 1):
    D[k, k] = 1
    D[m + 1 + k, m + 1 + k] = -1

# third and fourth blocks
for k in range(m):
    D[2 * m + 2 + k, 2 * m + 2 + k] = 1j
    D[3 * m + 2 + k, 3 * m + 2 + k] = -1j

# ----------------------------
# Check final relation F*V ≈ V*D
# ----------------------------
diff = F @ V - V @ D
norm_diff = np.linalg.norm(diff)
max_error = np.max(np.abs(diff))
is_zero = np.allclose(F @ V, V @ D, atol=1e-12)

# ----------------------------
# Diagnostics
# ----------------------------
print("----- MATRIX SHAPES -----")
print("X2 shape:", X2.shape)
print("Y2 shape:", Y2.shape)
print("T2 shape:", T2.shape)
print("Z2 shape:", Z2.shape)
print("V shape:", V.shape)
print("F shape:", F.shape)
print("D shape:", D.shape)

print("\n----- NUMERICAL CHECK -----")
print("‖F@V - V@D‖ =", norm_diff)
print("Max abs error =", max_error)
print("Is difference ≈ 0 ?", is_zero)


----- MATRIX SHAPES -----
X2 shape: (14, 4)
Y2 shape: (14, 4)
T2 shape: (14, 3)
Z2 shape: (14, 3)
V shape: (14, 14)
F shape: (14, 14)
D shape: (14, 14)

----- NUMERICAL CHECK -----
‖F@V - V@D‖ = 8.030405945646309e-15
Max abs error = 2.4183793897187097e-15
Is difference ≈ 0 ? True


#$theta=N-1$

In [22]:
import numpy as np

# ----------------------------
# Parameters
# ----------------------------
m = 3
N = 4 * m
t = N - 1
w = np.exp(-2j * np.pi / N)

# ----------------------------
# Construct C and S matrices
# ----------------------------
C = np.zeros((2*m, 2*m), dtype=complex)
for k in range(2*m):
    for l in range(2*m):
        C[k, l] = (1/np.sqrt(N)) * np.cos((k - t/2) * (l - t/2) * 2 * np.pi / N)

S = np.zeros((2*m, 2*m), dtype=complex)
for k in range(2*m):
    for l in range(2*m):
        S[k, l] = (1/np.sqrt(N)) * np.sin((k - t/2) * (l - t/2) * 2 * np.pi / N)

# ----------------------------
# Define X, Y, Z, T
# ----------------------------
X = C + 0.5 * np.eye(2*m)
Y = C - 0.5 * np.eye(2*m)
Z = S + 0.5 * np.eye(2*m)
T = S - 0.5 * np.eye(2*m)

# ----------------------------
# Construct shifted DFT matrix G
# ----------------------------
G = np.zeros((N, N), dtype=complex)
for k in range(N):
    for l in range(N):
        G[k, l] = (1/np.sqrt(N)) * w**((k - t/2) * (l - t/2))

DCT = G.real
DST = G.imag

# ----------------------------
# Helper matrices
# ----------------------------
J = np.flip(np.eye(2*m), axis=0)

X1 = X[:, :m]
Y1 = Y[:, :m]
Z1 = Z[:, :m]
T1 = T[:, :m]

# ----------------------------
# Build extended matrices
# ----------------------------
X2 = np.vstack([X1, J @ X1])
Y2 = np.vstack([Y1, J @ Y1])
Z2 = np.vstack([Z1, -J @ Z1])
T2 = np.vstack([T1, -J @ T1])

# ----------------------------
# Optional check like MATLAB: DST*Z2 + Z2
# ----------------------------
check_DST_Z2 = DST @ Z2 + Z2

# ----------------------------
# Combine all into V
# ----------------------------
V = np.hstack([X2, Y2, T2, Z2])

# ----------------------------
# Define diagonal matrix D
# ----------------------------
D = np.zeros((N, N), dtype=complex)
for k in range(m):
    D[k, k] = 1
    D[m + k, m + k] = -1
    D[2*m + k, 2*m + k] = 1j
    D[3*m + k, 3*m + k] = -1j

# ----------------------------
# Check final relation G*V ≈ V*D
# ----------------------------
diff = G @ V - V @ D
norm_diff = np.linalg.norm(diff)
max_error = np.max(np.abs(diff))
is_zero = np.allclose(G @ V, V @ D, atol=1e-12)

# ----------------------------
# Diagnostics (like previous code)
# ----------------------------
print("----- MATRIX SHAPES -----")
print("X2 shape:", X2.shape)
print("Y2 shape:", Y2.shape)
print("T2 shape:", T2.shape)
print("Z2 shape:", Z2.shape)
print("V shape:", V.shape)
print("G shape:", G.shape)
print("D shape:", D.shape)
print("DST*Z2 + Z2 shape:", check_DST_Z2.shape)

print("\n----- NUMERICAL CHECK -----")
print("‖G@V - V@D‖ =", norm_diff)
print("Max abs error =", max_error)
print("Is difference ≈ 0 ?", is_zero)


----- MATRIX SHAPES -----
X2 shape: (12, 3)
Y2 shape: (12, 3)
T2 shape: (12, 3)
Z2 shape: (12, 3)
V shape: (12, 12)
G shape: (12, 12)
D shape: (12, 12)
DST*Z2 + Z2 shape: (12, 3)

----- NUMERICAL CHECK -----
‖G@V - V@D‖ = 3.677002770555081e-15
Max abs error = 9.861271875154848e-16
Is difference ≈ 0 ? True


#$\theta=2N-1$

In [49]:
import numpy as np

# ----------------------------
# Parameters
# ----------------------------
m = 3
N = 4 * m
t = 2 * N - 1
w = np.exp(-2j * np.pi / N)

# ----------------------------
# Construct C and S matrices
# ----------------------------
C = np.zeros((2*m, 2*m), dtype=complex)
S = np.zeros((2*m, 2*m), dtype=complex)
for k in range(2*m):
    for l in range(2*m):
        C[k, l] = (1/np.sqrt(N)) * np.cos((k + 0.5) * (l + 0.5) * 2 * np.pi / N)
        S[k, l] = (1/np.sqrt(N)) * np.sin((k + 0.5) * (l + 0.5) * 2 * np.pi / N)


# ----------------------------
# Define X, Y, Z, T
# ----------------------------
X = C + 0.5 * np.eye(2*m)
Y = C - 0.5 * np.eye(2*m)
Z = S + 0.5 * np.eye(2*m)
T = S - 0.5 * np.eye(2*m)

# ----------------------------
# Construct shifted DFT matrix G
# ----------------------------
G = np.zeros((N, N), dtype=complex)
for k in range(N):
    for l in range(N):
        G[k, l] = (1/np.sqrt(N)) * w**((k - t/2) * (l - t/2))

# ----------------------------
# Helper matrix
# ----------------------------
J = np.flip(np.eye(2*m), axis=0)
X1 = X[:, :m]; Y1 = Y[:, :m]; Z1 = Z[:, :m]; T1 = T[:, :m]

# ----------------------------
# Build extended matrices
# ----------------------------
X2 = np.vstack([X1, -J @ X1])
Y2 = np.vstack([Y1, -J @ Y1])
Z2 = np.vstack([Z1, J @ Z1])
T2 = np.vstack([T1, J @ T1])

# ----------------------------
# Combine all into V
# ----------------------------
V = np.hstack([X2, Y2, T2, Z2])

# ----------------------------
# Define diagonal matrix D
# ----------------------------
D = np.zeros((N, N), dtype=complex)
for k in range(m):
    D[k, k] = 1
    D[m + k, m + k] = -1
    D[2*m + k, 2*m + k] = 1j
    D[3*m + k, 3*m + k] = -1j

# ----------------------------
# Check final relation G*V ≈ V*D
# ----------------------------
diff = G @ V - V @ D
diff[np.abs(diff) < 1e-14] = 0  # small values set to zero

# Only check equality
is_equal = np.all(diff == 0)
print("G @ V equals V @ D? ->", is_equal)


G @ V equals V @ D? -> True
